# Technical Note P1: Python Computational Grammar

**Wook Kang · Revised English edition · Version 1.0.0 · 3 October 2026**

This note connects a mathematical task to a numerical object, an implementation, and evidence about the result. It is an instructional computational document. Demonstration parameters are illustrative unless explicitly stated otherwise.

Run from a fresh kernel in cell order. See [the series README](../README.md) for setup and [editorial and verification notes](../docs/EDITORIAL_NOTES.md) for scope and limitations. The English prose has been reorganized and expanded from the supplied Korean notebook; this is an adapted edition rather than a line-by-line translation.

**Reading question:** What is given, what is unknown, what operation relates them, and what independent check can assess the output?

## Core mathematical structure

$$
\boxed{
\text{data}
+
\text{functions}
+
\text{control flow}
+
\text{objects}
+
\text{modules}
}
$$

In [1]:
# Reproducible display and scale-aware comparison.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
def scaled_allclose(a, b, rtol=1e-12, atol=None, **kwargs):
    """Compare numerical examples using an absolute tolerance scaled to b."""
    a, b = np.asarray(a), np.asarray(b)
    scale = float(np.max(np.abs(b))) if b.size else 0.0
    if atol is None:
        atol = 1e-14 * max(scale, np.finfo(float).tiny)
    return np.allclose(a, b, rtol=rtol, atol=atol, **kwargs)


## 1. A computational statement

Read a function call as a mapping from inputs and parameters to an output. Assignment binds a name to an object; it does not assert a mathematical equality.

$$
y=f(x;\theta)
$$

In [2]:
def model(x, theta):
    return theta * x**2

x = 3.0
theta = 2.0
y = model(x, theta)

print("y =", y)

y = 18.0


```python
x = 3.0
```

$$
x=3
$$

## 2. Basic Python objects

Integers represent counts and indices; floats represent approximate numbers; booleans control decisions; strings carry labels; containers collect related objects.

| Python object | Example | Use |  
| --- | --- | --- |  
| `int` | `10` | index, count |  
| `float` | `1.0e-10` | parameter, physical quantity |  
| `bool` | `True` | option, switch |  
| `str` | `"Robin"` | label, method name |  
| `list` | `[1,2,3]` | ordered mutable collection |  
| `tuple` | `(Nx,Ny)` | fixed structure, shape-like data |  
| `dict` | `{"D":1e-10}` | named parameters/configuration |  
| `set` | `{"FDM","FVM"}` | unique items |

$$
\boxed{
\texttt{list},\quad
\texttt{tuple},\quad
\texttt{dict}
}
$$

In [3]:
D = 1.0e-10
method = "FVM"
shape = (40, 20)

params = {
    "D": D,
    "method": method,
    "shape": shape,
}

print(params)
print("D =", params["D"])

{'D': 1e-10, 'method': 'FVM', 'shape': (40, 20)}
D = 1e-10


## 3. Mutable and immutable objects

Assignment can make two names refer to the same mutable object. A tuple is immutable as a container, but a mutable object stored inside it can still change.

In [4]:
a = [1, 2, 3]
b = a

b[0] = 100

print("a =", a)
print("b =", b)

a = [100, 2, 3]
b = [100, 2, 3]


$$
\boxed{
\text{assignment}
\neq
\text{deep copy}
}
$$

## 4. Lists, tuples, and dictionaries

Use a list for an ordered collection, a tuple for a fixed grouping, and a dictionary for named settings. None of these choices automatically enforces units or physical admissibility.

```python
times = [0, 1, 2, 4, 8]
```

```python
shape = (Nx, Ny, Nz)
```

```python
material = {
    "rho": 500.0,
    "D": 1e-10
}
```

$$
\boxed{
\begin{array}{ll}
\text{ordered data} &\rightarrow \texttt{list}\\
\text{fixed structure} &\rightarrow \texttt{tuple}\\
\text{named configuration} &\rightarrow \texttt{dict}
\end{array}
}
$$

In [5]:
times = [0, 1, 2, 4, 8]
shape = (60, 30, 12)

material = {
    "density": 520.0,
    "diffusivity": 1.0e-10,
}

print("times:", times)
print("shape:", shape)
print("density:", material["density"])

times: [0, 1, 2, 4, 8]
shape: (60, 30, 12)
density: 520.0


## 5. Conditional branches

An if statement can select a regime or a piecewise constitutive relation. State the threshold, equality case, and valid domain explicitly.

In [6]:
bc_type = "Robin"

if bc_type == "Dirichlet":
    print("Use prescribed boundary value.")
elif bc_type == "Neumann":
    print("Use prescribed boundary flux.")
elif bc_type == "Robin":
    print("Use surface-transfer law.")
else:
    raise ValueError("Unknown boundary condition.")

Use surface-transfer law.


```python
if ...
elif ...
else ...
```

$$
\text{case 1},\quad
\text{case 2},\quad
\text{case 3}
$$

## 6. Boolean expressions

A condition evaluates to True or False. For arrays, use elementwise logical operations with parentheses, then reduce with all or any when a scalar decision is needed.

$$
D>0
$$

In [7]:
D = 1.0e-10

print(D > 0)
print(D <= 0)

if D <= 0:
    raise ValueError("Diffusivity must be positive.")
else:
    print("Positivity check passed.")

True
False
Positivity check passed.


```python
if dt > 0 and nsteps >= 1:
    ...
```

$$
(\Delta t>0)\land(N_{\mathrm{step}}\ge1)
$$

## 7. For loops

Loops express time steps, parameter sweeps, and nonlinear iterations. Distinguish sequential updates from independent spatial arithmetic that can be vectorized.

```python
for ...
```

In [8]:
times = [0, 1, 2, 4, 8]

for t in times:
    print("time =", t)

time = 0
time = 1
time = 2
time = 4
time = 8


$$
u^{n+1}=
u^n+\Delta t\,f(u^n)
$$

In [9]:
u = 1.0
dt = 0.1
k = 0.5

for n in range(5):
    u = u + dt*(-k*u)
    print(f"step={n+1}, u={u:.6f}")

step=1, u=0.950000
step=2, u=0.902500
step=3, u=0.857375
step=4, u=0.814506
step=5, u=0.773781


## 8. Range, enumerate, and zip

Range supplies integer indices, enumerate pairs positions with values, and zip pairs sequences. By default zip stops at the shortest input; use strict=True when equal lengths are required.

```python
for i in range(N):
```

```python
for i, value in enumerate(values):
```

```python
for t, y in zip(times, observations):
```

In [10]:
times = [0, 1, 2]
obs = [0.30, 0.27, 0.24]

for i, (t, y) in enumerate(zip(times, obs)):
    print(i, t, y)

0 0 0.3
1 1 0.27
2 2 0.24


## 9. While loops and convergence

A residual-controlled loop also needs an iteration limit, finite-value checks, and an explicit failure outcome. Stopping because the budget is exhausted is not convergence.

$$
\|R(u^{(k)})\|>\varepsilon
$$

In [11]:
u = 1.0
tol = 1e-6
max_iter = 50

k = 0
while abs(u) > tol and k < max_iter:
    u *= 0.5
    k += 1

print("iterations =", k)
print("u =", u)

iterations = 20
u = 9.5367431640625e-07


$$
\boxed{
\text{convergence criterion}
+
\text{iteration limit}
}
$$

## 10. Functions as mathematical interfaces

A function can represent a material law, source, residual, initial condition, or objective. Make its inputs, output shape, units, and side effects clear.

$$
y=f(x)
$$

In [12]:
def diffusivity(u, a=1e-10, b=5.0, u_ref=0.20):
    return a * (2.718281828459045 ** (b*(u-u_ref)))

print(diffusivity(0.25))

1.2840254166877413e-10


$$
\boxed{
\text{function}
\approx
\text{mathematical relation / operator}
}
$$

## 11. Positional and keyword arguments

Named arguments expose the meaning of a call and reduce confusion between similar parameters. They are particularly useful for solver tolerances and material settings.

```python
diffusivity(0.25, 1e-10, 5.0, 0.20)
```

```python
diffusivity(
    u=0.25,
    a=1e-10,
    b=5.0,
    u_ref=0.20
)
```

In [13]:
def robin_flux(u_surface, u_env, h):
    return h * (u_surface - u_env)

q = robin_flux(
    u_surface=0.25,
    u_env=0.10,
    h=2e-8,
)

print("q =", q)

q = 3e-09


## 12. Default arguments

Defaults are evaluated when the function is defined. Use None followed by initialization inside the function when a fresh mutable container is needed on each call.

```python
def f(x, tol=1e-8):
```

```python
def add_item(x, items=[]):
    ...
```

```python
def add_item(x, items=None):
    if items is None:
        items = []
```

In [14]:
def add_item(x, items=None):
    if items is None:
        items = []
    items.append(x)
    return items

print(add_item(1))
print(add_item(2))

[1]
[2]


## 13. Functions are objects

Pass the equation as a callable rather than a precomputed value. SciPy repeatedly evaluates that callable at trial states selected by the algorithm.

In [15]:
def square(x):
    return x*x

def apply_function(f, x):
    return f(x)

print(apply_function(square, 3.0))

9.0


```python
solver(f)
```

$$
\boxed{
f(x)=0
\quad\Longrightarrow\quad
\text{pass }f\text{ to a solver}
}
$$

## 14. Lambda expressions

A lambda is useful for a short callback. Use a named function for a substantial residual or constitutive law so its assumptions can be documented and tested.

$$
f(x)=x^2-2
$$

In [16]:
f = lambda x: x**2 - 2

print(f(1.5))

0.25


## 15. Scope

Python resolves names through local, enclosing, global, and built-in scopes. Explicit parameters are easier to trace than a solver that depends on mutable global state.

In [17]:
a = 10.0   # global

def example(x):
    b = 2.0   # local
    return a*x + b

print(example(3.0))

32.0


$$
\boxed{
\text{input}
\rightarrow
\text{function}
\rightarrow
\text{output}
}
$$

## 16. Return multiple results

A function can return a tuple containing the solution, iteration count, and status. The caller should inspect the status rather than extracting only a number.

In [18]:
def evaluate_model(x):
    y = x**2
    dy = 2*x
    return y, dy

value, derivative = evaluate_model(3.0)

print("value =", value)
print("derivative =", derivative)

value = 9.0
derivative = 6.0


```python
return residual, jacobian
```

```python
return solution, iteration_count
```

```python
return field, flux
```

## 17. Unpacking

Unpacking assigns names to elements of a returned structure. Match the number and meaning of outputs to the function interface.

```python
a, b = (1, 2)
```

```python
Nx, Ny, Nz = shape
```

In [19]:
shape = (60, 30, 12)

Nx, Ny, Nz = shape

print(Nx, Ny, Nz)

60 30 12


## 18. List comprehensions

A comprehension builds a collection from an iteration. It improves compactness but does not itself provide NumPy-style numerical vectorization.

In [20]:
values = []
for x in range(5):
    values.append(x*x)

print(values)

[0, 1, 4, 9, 16]


In [21]:
values = [x*x for x in range(5)]
print(values)

[0, 1, 4, 9, 16]


## 19. Classes and objects

A class groups state and operations when that grouping clarifies the model. Small scientific programs can often remain clear with functions and explicit parameter containers.

In [22]:
class Material:
    def __init__(self, name, density, D):
        self.name = name
        self.density = density
        self.D = D

    def diffusivity(self, u):
        return self.D

wood = Material(
    name="example wood",
    density=520.0,
    D=1e-10,
)

print(wood.name)
print(wood.diffusivity(0.25))

example wood
1e-10


$$
\boxed{
\text{object}=
\text{state}
+
\text{behavior}
}
$$

$$
\boxed{
\text{function + dict}
}
$$

## 20. Dataclasses

A dataclass provides a named parameter container with generated initialization. Type annotations document intent but do not automatically validate ranges or units.

In [23]:
from dataclasses import dataclass

@dataclass
class SimulationConfig:
    Lx: float
    Nx: int
    dt: float
    nsteps: int

cfg = SimulationConfig(
    Lx=0.05,
    Nx=60,
    dt=1800.0,
    nsteps=8,
)

print(cfg)
print("total time =", cfg.dt * cfg.nsteps)

SimulationConfig(Lx=0.05, Nx=60, dt=1800.0, nsteps=8)
total time = 14400.0


```python
cfg["dt"]
```

```python
cfg.dt
```

## 21. Modules and imports

Move reusable definitions into modules as the project grows. Keep model definitions, execution, and reporting sufficiently separate to allow independent testing.

```text
model.py
solver.py
boundary.py
postprocess.py
```

```python
import solver
```

```python
from solver import solve
```

$$
\boxed{
\text{one huge script}
\rightarrow
\text{small reusable modules}
}
$$

## 22. The standard library

Pathlib handles paths, json records structured data, time measures elapsed time, and dataclasses organize configuration. Prefer these tools to platform-specific path strings.

| module | Use |  
| --- | --- |  
| `math` | scalar mathematical functions |  
| `pathlib` | file path |  
| `csv` | CSV |  
| `json` | structured text data |  
| `time` | timing |  
| `dataclasses` | configuration objects |  
| `typing` | type hints |  
| `copy` | shallow/deep copy |

In [24]:
from pathlib import Path
from time import perf_counter

p = Path("results") / "output.csv"

t0 = perf_counter()
total = sum(i*i for i in range(100000))
elapsed = perf_counter() - t0

print("path =", p)
print("elapsed =", elapsed)

path = results/output.csv
elapsed = 0.004146678000324755


## 23. Exceptions

Reject invalid inputs and report numerical failure explicitly. Catch only exceptions that the surrounding code can meaningfully handle.

In [25]:
def checked_diffusivity(D):
    if D <= 0:
        raise ValueError("D must be positive.")
    return D

try:
    checked_diffusivity(-1e-10)
except ValueError as e:
    print("Caught error:", e)

Caught error: D must be positive.


## 24. Assertions

Assertions check internal assumptions during development. They may be disabled by optimized Python execution, so use explicit exceptions for required input validation.

In [26]:
shape = (20, 30)

assert len(shape) == 2
assert all(n > 0 for n in shape)

print("shape checks passed")

shape checks passed


## 25. Type hints

Annotations improve documentation, editor assistance, and static checking. A float annotation cannot distinguish metres from millimetres or guarantee positivity.

In [27]:
def decay_step(u: float, k: float, dt: float) -> float:
    return u - dt*k*u

print(decay_step(1.0, 0.5, 0.1))

0.95


## 26. Pure functions and side effects

A pure numerical mapping is easier to test and reuse in an inverse solver. File writes, mutations, and hidden random draws are side effects that should be visible.

In [28]:
def pure_flux(D, grad_u):
    return -D * grad_u

print(pure_flux(2.0, 3.0))

-6.0


```python
history.append(value)
```

## 27. Configuration and state

Configuration describes geometry, material settings, and solver choices. State contains evolving temperatures, moisture values, displacements, or iterates; keep their roles distinct.

$$
\boxed{
\text{configuration}
\neq
\text{state}
}
$$

In [29]:
config = {
    "dt": 0.1,
    "k": 0.5,
}

state = {
    "time": 0.0,
    "u": 1.0,
}

for step in range(3):
    state["u"] = state["u"] - config["dt"]*config["k"]*state["u"]
    state["time"] += config["dt"]

print("config =", config)
print("state =", state)

config = {'dt': 0.1, 'k': 0.5}
state = {'time': 0.30000000000000004, 'u': 0.857375}


## 28. A small scientific program

Define settings, initialize the state, evaluate the model, advance or solve, check the result, and record output. This structure persists when a scalar state becomes a large field.

```text
1. define parameters
2. define model functions
3. initialize state
4. iterate / solve
5. compute diagnostics
6. return / save / plot
```

In [30]:
def rhs(u, k):
    return -k*u

def simulate(u0, k, dt, nsteps):
    u = u0
    history = [(0.0, u)]

    for n in range(nsteps):
        u = u + dt*rhs(u, k)
        t = (n+1)*dt
        history.append((t, u))

    return history

history = simulate(
    u0=1.0,
    k=0.5,
    dt=0.1,
    nsteps=5,
)

for t, u in history:
    print(f"t={t:.2f}, u={u:.6f}")

t=0.00, u=1.000000
t=0.10, u=0.950000
t=0.20, u=0.902500
t=0.30, u=0.857375
t=0.40, u=0.814506
t=0.50, u=0.773781


$$
\frac{du}{dt}=-ku
$$

$$
\frac{\partial u}{\partial t}=
\nabla\cdot(D\nabla u)
$$

## 29. The main guard

Code under if __name__ == '__main__' runs when a script is executed directly. Definitions can then be imported without automatically launching the calculation.

```python
if __name__ == "__main__":
    ...
```

## 30. The boundary between Python and NumPy

Keep control loops when successive steps depend on earlier states. Use array operations for independent numerical work; clarity and memory use matter as well as speed.

```python
for i in range(N):
    ...
```

$$
\boxed{
\text{Python collection}
\rightarrow
\text{NumPy ndarray}
}
$$

$$
\boxed{
\text{explicit Python loop}
\rightarrow
\text{array operation / vectorization}
}
$$

## 31. How to read unfamiliar scientific code

Find inputs, state variables, model callables, update loops, stopping conditions, and output checks. Follow these dependencies before interpreting individual expressions.

$$
\boxed{
\text{input}
\rightarrow
\text{operator}
\rightarrow
\text{state update}
\rightarrow
\text{output}
}
$$

## 32. Computational grammar map

A name binds an object, a function defines a mapping, a branch selects a case, and a loop repeats an operation. Syntax acquires meaning through the mathematical problem.

| Mathematical or computational concept | Python syntax |  
| --- | --- |  
| Variable or parameter | name |  
| Function $f(x)$ | `def` |  
| Case selection | `if / elif / else` |  
| Repeated operation | `for / while` |  
| Convergence condition | boolean expression |  
| parameter set | `dict` / `dataclass` |  
| state | mutable object |  
| mapping | function |  
| model family | function/class |  
| Failure condition | exception |  
| Code reuse | module/import |  
| Pass a function to a solver | function object / callable |  
| Grouped results | tuple/dict/object |

## 33. Five ideas to retain

Keep interfaces explicit, track mutation, pass callables to solvers, separate state from configuration, and report failure. These habits matter before choosing an advanced algorithm.

## 34. Continue to P2

The next step is to translate scalars, components, and sampled fields into arrays with declared shapes, axes, and dtypes.

$$
\boxed{
\text{scalar / vector / matrix / field}
\rightarrow
\texttt{ndarray}
}
$$

## Further work

1. Change one parameter or discretization choice and predict the effect before running.
2. Write the input and output shapes, units, and field locations.
3. Construct a deliberately invalid input and make the calculation report it clearly.
4. Separate an execution check from an accuracy check and from any physical claim.

## Primary documentation

- [Python language reference](https://docs.python.org/3/reference/)

These are living documentation links. The accompanying requirements and verification report identify the versions actually executed for this edition.

## AI assistance and responsibility

The English adaptation, editorial supplementation, and code review were prepared with AI assistance. Wook Kang retains responsibility for reviewing the explanations, examples, and any subsequent research application. No new experimental validation is claimed.